# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Compute the baseline label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Baseline: majority class prediction (54.2% positive class)
baseline_prob = df['is_declining_label'].mean()
baseline_pred = (df['is_declining_label'] == 1).astype(int)
baseline_accuracy = (baseline_pred == df['is_declining_label']).mean()

print('Baseline probability of decline:', round(baseline_prob, 4))
print('Baseline accuracy (always predict decline):', round(baseline_accuracy, 4))

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Add action score: probability of decline
df['action_score'] = baseline_prob

# Rank by action score descending (highest decline risk first)
df = df.sort_values('action_score', ascending=False).reset_index(drop=True)
df['rank'] = range(1, len(df) + 1)

# Write the ranked CSV (create outputs dir if needed)
import os
os.makedirs('work/outputs', exist_ok=True)
df[['content_id', 'client_id', 'action_score', 'rank']].to_csv('work/outputs/baseline_action_score.csv', index=False)

print('Wrote work/outputs/baseline_action_score.csv')
print('Total pages ranked:', len(df))
print('Top 5 action scores:')
print(df.head(5)[['content_id', 'action_score', 'rank']].to_string())

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
df = pd.read_csv('work/outputs/baseline_action_score.csv')

top20 = df.head(20)

for i in range(len(top20)):
    row = top20.iloc[i]
    rank = row['rank']
    score = row['action_score']
    cid = row['content_id']
    reasons = []
    if row.get('impressions_last_30d', 0) < df['impressions_last_30d'].median():
        reasons.append('low_prev30d_impressions')
    if row.get('engagement_rate', 0) < df['engagement_rate'].median():
        reasons.append('low_engagement_rate')
    if row.get('ctr', 0) > df['ctr'].median():
        reasons.append('high_ctr_vs_base')
    print(f'Rank {rank}: content_id={cid}, score={score:.4f}, reasons={reasons}')
    conf = 'high' if score > 0.6 else 'medium'
    print(f'  Confidence: {conf}')
    print(f'  Could be wrong: page stabilizes or gains traction in next 30d')
    print()

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

- Check: are any top-20 pages receiving high action scores because of `impressions_last_30d`?
  The last-30d window overlaps the label period — using it would be leakage. Only `*_prev30d` columns are safe.
- Check: no product flags or existing-system scores appear as drivers of the action score.
- Check: `trend_direction` and `trend_pct` are NOT used as features (they are label-derived).
- Expected: top picks are driven by systematic low-engagement signals (low engagement_rate, low prev30d metrics), not forward-looking data.

## Self-check

Before you submit, confirm each line honestly:
- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.